In [8]:
import napari
import nd2
import os
import numpy as np
import dask.array as da

from scipy.ndimage import convolve
from tifffile import imwrite

from tqdm.dask import TqdmCallback

In [2]:
path = "/mnt/z/Dasha/2. FAST-AB biosensors/Microscope/Inverted bladder model/TrimFABS sensor 30.06.2026/To analyze"

file = "NCCRBM128_sample4.nd2"

files = [
    "NCCRBM128_sample4_before.nd2",
    "NCCRBM128_sample4_TMP.nd2",
]

imgs = [
    nd2.ND2File(os.path.join(path, file))
    for file in files
]

img_da = [img.to_dask() for img in imgs]


file = "NCCRBM128_sample5.nd2"
    
nd2_file = nd2.ND2File(os.path.join(path, file))
tmp = nd2_file.to_dask()

img_da.extend([tmp[i] for i in range(2)])


file = "NCCRBM128_sample6.nd2"
    
nd2_file = nd2.ND2File(os.path.join(path, file))
tmp = nd2_file.to_dask()

img_da.extend([tmp[i] for i in range(2)])

img = da.stack(img_da, axis=0)

In [3]:
with TqdmCallback():
    mean = img.mean(axis=(0, 1, 2)).compute()
    #std = img.std(axis=(0, 1, 2)).compute()

  0%|          | 0/1816 [00:00<?, ?it/s]

In [4]:
imwrite("/mnt/f/pixel-mean.tif", mean)
#imwrite("/mnt/f/pixel-std.tif", std)

In [6]:
with TqdmCallback():
    var = ((img-mean)**2).mean(axis=(0, 1, 2)).compute()

  0%|          | 0/2656 [00:00<?, ?it/s]

In [7]:
imwrite("/mnt/f/pixel-var.tif", var)

In [13]:
weights = np.ones((7, 7))
weights[3, 3] = 0

conv = np.empty_like(mean)
for i in range(len(conv)):
    conv[i] = convolve(mean[i], weights) / weights.sum()

In [14]:
imwrite("/mnt/f/pixel-conv.tif", np.abs(mean - conv))